# What does one bin do to a pure tone?

`SpectralPlan` names the segments: their length, their starts, and the frequency each bin is aimed at. This notebook asks the next question. Once a segment length and a window are fixed, what complex number does that bin assign to a pure tone?

The coefficient is the direct sum after the detrend and the window,

$$X = \sum_n y[n]\, e^{-i\omega n}.$$

The sum is left unscaled. The one-sided factor of 2 appears when a density is formed. For `order=-1` the equivalent noise bandwidth is

$$\mathrm{ENBW} = f_s S_2 / S_1^2,$$

with $S_1 = \sum w$ and $S_2 = \sum w^2$. $S_1$ is the sum of the window samples, and $S_2$ is the sum of their squares. A detrend replaces that formula with the ENBW of the window after the polynomial has been removed. `SpectrumResult.ENBW` stays on the raw-window formula either way.

The cross-spectrum is $X Y^{*}$. A tone on channel 1 that lags channel 0 by an integer number of samples therefore has cross-spectrum phase $+\omega\tau$. The transfer-function phase is $-\omega\tau$. Here $\tau$ is that delay in seconds, and $\omega = 2\pi f$.


In [ ]:
import numpy as np

from speckit.response import BinResponse

L = 256
fs = 256.0
freq = 32.0
window = np.hanning(L)
response = BinResponse(window, fs, freq, order=-1)
formula = fs * response.S2 / response.S1**2
print(f"window ENBW {response.window_enbw:.6g} Hz")
print(f"DFT integral {response.enbw_numerical:.6g} Hz")
print(f"formula fs*S2/S1^2 {formula:.6g} Hz")
r = fs / L
for offset in (0.0, 0.5 * r, 3.0 * r, -2.0 * freq):
    coef = response.coefficient(freq + offset)
    print(f"offset {offset:+.4g} Hz  |X|/S1 {abs(coef) / response.S1:.6g}")


In [ ]:
from speckit import compute_single_bin

n = np.arange(L)
omega = 2.0 * np.pi * freq / fs
delay = 3
x = np.cos(omega * n)
y = np.cos(omega * (n - delay))
# One segment, rectangular window: integer-bin leakage of the other
# frequency is a zero of the Dirichlet kernel, so the phase is the delay.
tone = compute_single_bin(
    np.vstack([x, y]),
    fs,
    freq=freq,
    L=L,
    order=-1,
    win=np.ones,
    olap=0.0,
    storage="replicates",
)
print(f"phase(X conj(Y)) {np.angle(tone.XY[0]):+.6g} rad")
print(f"+omega * delay   {+(omega * delay):+.6g} rad")
print(f"phase(Hxy)       {np.angle(tone.Hxy[0]):+.6g} rad")
print(f"mean |Xseg|^2    {np.mean(np.abs(tone.Xseg)**2):.6g}")
print(f"summary XX       {float(tone.XX[0]):.6g}")


## Reading the result

The first printout is three ways of stating the same ENBW. The window formula, the sum of the squared DFT of the window, and `BinResponse.enbw` agree for `order=-1`. The offset lines are the bin's gain relative to $S_1$. Offset 0 is the coherent gain, the response to a tone at the analysis frequency. Half a resolution bandwidth is still inside the main lobe. Three resolution bandwidths is out in the sidelobes. The last offset is the negative-frequency image, `coefficient(-freq)`.

The second cell is a cosine at an integer bin, and the same cosine delayed by 3 samples. `Gxy = X * conj(Y)`, so the cross-spectrum phase is $+\omega\tau$. `Hxy = conj(XY) / XX` has phase $-\omega\tau$. `storage="replicates"` keeps one coefficient per segment. The mean of $|X_{\mathrm{seg}}|^2$ is the summary `XX`. Those replicates stay overlapped when the plan overlaps them. `nu` is the number of independent pieces those overlapped segments still carry. It equals `2K` when the segments share no samples. `K` is the number of segments.
